# 06f — Flow matching

A learned transport from `N(0, I)` onto `p(u_{n+1} | u_n)` -- no assumption about the shape of
the conditional, where the Gaussian rung assumes a diagonal normal. Trained by conditional flow
matching on straight-line paths; sampled by integrating the learned velocity field.

Not tuned, and solving a strictly harder problem on the same budget: a poor number here means
"not with this budget", not "flow matching does not work".

In [1]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks'
                       else pathlib.Path.cwd()))
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt, torch
from l63 import ARTIFACTS, evaluate as E, plots as P
from l63.data import make_datasets
from run.report import clean, load_model, load_rows, summarise

gt = json.load(open(ARTIFACTS / 'ground_truth.json'))
S  = clean(summarise(load_rows()))
DATA = list(gt['datasets'])                      # 'ode', 'sde', 'sde015'

def num(v, w=6, p=2):
    """A ruler value, or an em dash where it is genuinely undefined."""
    if isinstance(v, dict):
        v = v.get('median')
    undefined = v is None or v != v          # None from clean(), bare NaN from the json
    return f'{"—":>{w}}' if undefined else f'{v:{w}.{p}f}'

def rng(v, p=2):
    if v is None or v.get('lo') is None or v['lo'] != v['lo']:
        return '—'
    return f"[{v['lo']:.{p}f}–{v['hi']:.{p}f}]"

print(len(S), 'model x dataset entries ·', len(DATA), 'datasets')

43 model x dataset entries · 3 datasets


## Numbers

In [2]:
for key in DATA:
    print(f"--- {key.upper()} ---")
    for name, tag in ((f'06f_flow_{key}', 'flow'), (f'06_gaussian_{key}', 'gaussian')):
        s = S.get(name)
        if s is None: continue
        print(f"  {tag:10s} horizon {num(s['horizon'],4,0)}   climate {num(s['climate'])}   "
              f"spread {num(s['spread'])}   alive {num(s['alive'])}   {rng(s['alive'])}")
    s = S.get(f'06f_flow_{key}')
    if s:
        m = s['mean_only']
        print(f"  {'flow x0=0':10s} horizon {num(m['horizon'],4,0)}   climate {num(m['climate'])}   "
              f"spread {num(m['spread'])}   alive {num(m['alive'])}   {rng(m['alive'])}")

--- ODE ---
  flow       horizon  270   climate  16.48   spread      —   alive   1.00   [1.00–1.00]
  gaussian   horizon  240   climate  18.58   spread      —   alive   1.00   [1.00–1.00]
  flow x0=0  horizon  275   climate  21.54   spread      —   alive   1.00   [0.56–1.00]
--- SDE ---
  flow       horizon   27   climate  20.78   spread   0.88   alive   1.00   [1.00–1.00]
  gaussian   horizon   27   climate   5.18   spread   1.00   alive   1.00   [1.00–1.00]
  flow x0=0  horizon   38   climate 176.89   spread   0.00   alive   0.00   [0.00–0.00]
--- SDE015 ---
  flow       horizon  115   climate  15.58   spread   0.75   alive   1.00   [1.00–1.00]
  gaussian   horizon   89   climate   6.50   spread   0.94   alive   1.00   [1.00–1.00]
  flow x0=0  horizon  115   climate  43.47   spread   0.00   alive   0.50   [0.00–1.00]


## This model

In [3]:
KEY = 'ode'      # any of DATA
s = S['06f_flow_' + KEY]
ref = gt['datasets'][KEY]
d = make_datasets(seed=0, kind=ref['kind'], b=ref['b'])
m, hist = load_model('06f_flow_' + KEY + '_s' + str(s['rep_seed']))

print(f"{m.n_params:,} parameters, history {m.history}, figures show seed {s['rep_seed']}")
print()
print(f"{'ruler':16s}{'median':>8s}   range over seeds")
for k in ('horizon', 'spread', 'climate', 'climate_vs_truth', 'chaos', 'alive', 'lobe'):
    v = s[k]
    p = 0 if k == 'horizon' else 2
    print(f"  {k:14s}{num(v, 8, p)}   {rng(v, p)} over {v['n']} seeds")
print(f"\ntruth on this dataset:  climate {ref['truth_climate']:.2f}   "
      f"alive {ref['truth_alive']:.2f}   lobe {ref['truth_lobe']:.2f}   "
      f"ground truth usable {ref['floor_steps']} steps")
print(f"\nfirst steps, ||u_hat_n - u_n|| in Lorenz units:")
for i, e in enumerate(s['early'][:6], 1):
    print(f"  n={i}  {e:.3e}")

34,435 parameters, history 1, figures show seed 1

ruler             median   range over seeds
  horizon            270   [154–334] over 5 seeds
  spread               —   — over 0 seeds
  climate          16.48   [5.68–34.30] over 5 seeds
  climate_vs_truth   26.57   [9.15–55.28] over 5 seeds
  chaos             0.94   [0.61–0.98] over 5 seeds
  alive             1.00   [1.00–1.00] over 5 seeds
  lobe              0.69   [0.54–0.73] over 5 seeds

truth on this dataset:  climate 0.62   alive 1.00   lobe 0.63   ground truth usable 362 steps

first steps, ||u_hat_n - u_n|| in Lorenz units:
  n=1  6.634e-02
  n=2  1.191e-01
  n=3  1.718e-01
  n=4  2.109e-01
  n=5  2.726e-01
  n=6  3.032e-01


## Figures

Banked by `run/report.py`; regenerated here from the same checkpoint so the notebook and the deck cannot disagree.

In [4]:
P.loss_figure(hist, None, n_val_traj=8); plt.show()
P.arch_figure(m.spec(), "", m.n_params, None); plt.show()
long = d.raw(m.forecast(d.eval[:gt['n_long'], :m.history], gt['long_steps']))
P.lorenz_map_figure(long, d.raw(d.eval), None); plt.show()

/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50783/4260566466.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.loss_figure(hist, None, n_val_traj=8); plt.show()
/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50783/4260566466.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.arch_figure(m.spec(), "", m.n_params, None); plt.show()


/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50783/4260566466.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.lorenz_map_figure(long, d.raw(d.eval), None); plt.show()


## Findings

_Written after reading the numbers above._

- 
- 
- 